# UEBA Portable — Entraînement sur Google Colab

Ce notebook entraîne l'ensemble de détection (IsolationForest + OneClassSVM + Autoencoder MLP)
sur la fixture synthétique et persiste le modèle dans Google Drive.

Compatible Google Colab (GPU non requis — calcul CPU suffisant pour les 16 features).

> **Données** : fixture synthétique incluse dans le dépôt — aucune donnée de production.
> Voir `tests/integration/fixtures/sample_logs.csv`.

In [ ]:
# Installation des dépendances (uniquement sur Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q python-dateutil pydantic scikit-learn numpy pandas
    !git clone https://github.com/YOUR_ORG/ueba-portable.git
    sys.path.insert(0, 'ueba-portable/src')
    FIXTURE = 'ueba-portable/tests/integration/fixtures/sample_logs.csv'
else:
    sys.path.insert(0, '../src')
    FIXTURE = '../tests/integration/fixtures/sample_logs.csv'

print(f'Environnement : {"Google Colab" if IN_COLAB else "Local"}')

## 1. Chargement et extraction des features

In [ ]:
import csv
from datetime import timedelta
from pathlib import Path

import numpy as np

from ueba.adapters.wazuh import WazuhAdapter
from ueba.domain.features import UEBAFeatureExtractor, FEATURE_NAMES

with Path(FIXTURE).open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

events = WazuhAdapter().normalize(records)
extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1),
    window_step=timedelta(minutes=30),
)
vectors = extractor.extract(events)

# Séparer les fenêtres normales (13 mai) pour l'entraînement
train_vectors = [v for v in vectors if v.window_start.day == 13]
X_train = np.array([v.to_vector() for v in train_vectors])

print(f'Vecteurs totaux     : {len(vectors)}')
print(f'Vecteurs train (J1) : {len(train_vectors)}')
print(f'Shape X_train       : {X_train.shape}')

## 2. Entraînement de l'ensemble

In [ ]:
from ueba.domain.ensemble import AnomalyEnsemble

ensemble = AnomalyEnsemble(
    n_estimators=100,
    contamination=0.05,
    svm_nu=0.05,
    autoencoder_hidden_layers=(8, 4, 8),
    random_state=42,
)
ensemble.fit(X_train)

print('Ensemble entraîné.')

## 3. Évaluation sur le scénario de spray (16 mai)

In [ ]:
spray_vectors = [
    v for v in vectors
    if v.window_start.day == 16 and v.window_start.hour == 14
    and v.failed_login_count > 0
]
X_spray = np.array([v.to_vector() for v in spray_vectors])

verdicts = ensemble.predict(X_spray)

print(f'Fenêtres spray évaluées : {len(spray_vectors)}')
for v, verdict in zip(spray_vectors, verdicts):
    status = '🚨 ANOMALIE' if verdict.is_anomaly else '✅ normal'
    print(f'  {v.user:20s}  score={verdict.anomaly_score:.2f}  {status}')

## 4. Persistance du modèle

In [ ]:
import os

# Sur Colab : monter Drive pour persister hors session
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_PATH = '/content/drive/MyDrive/ueba_models/ensemble.joblib'
else:
    MODEL_PATH = '../models/ensemble.joblib'

ensemble.save(MODEL_PATH)
print(f'Modèle sauvegardé : {MODEL_PATH}')

# Vérification du rechargement
from ueba.domain.ensemble import AnomalyEnsemble as AE
reloaded = AE.load(MODEL_PATH)
verdicts2 = reloaded.predict(X_spray)
assert all(v1.is_anomaly == v2.is_anomaly for v1, v2 in zip(verdicts, verdicts2))
print('Rechargement OK — résultats identiques.')

## 5. Mapping MITRE ATT&CK

In [ ]:
from ueba.domain.mitre import MitreMapper

mapper = MitreMapper()
pop_matches = mapper.match_population(spray_vectors)

for m in pop_matches:
    print(f'[{m.technique_id}] {m.technique_name}')
    print(f'  Tactique  : {m.tactic}')
    print(f'  Source    : {m.source}')
    print(f'  Rationale : {m.rationale}')